In [ ]:
import cdsapi
import os
import time

client = cdsapi.Client()

SAVE_DIR = r"path/to/snow/data-and-results\snow_era5_monthly"
os.makedirs(SAVE_DIR, exist_ok=True)
# Safe retrieve: retry = 0 sec
def safe_retrieve(dataset, request, target):
    while True:
        try:
            client.retrieve(dataset, request, target)
            return
        except Exception as e:
            print("\nDownload failed, retrying immediately...\n", e)
            continue
# Download a full month
def download_month(year, month):
    yyyy = f"{year:04d}"
    mm   = f"{month:02d}"

    fname = f"{yyyy}-{mm}.nc"
    fpath = os.path.join(SAVE_DIR, fname)

    if os.path.exists(fpath):
        print("Already exists:", fname)
        return

    print(f"\n=== Downloading {fname} ===")

    request = {
        "product_type": "reanalysis",
        "variable": ["2m_temperature"],
        "year": yyyy,
        "month": [mm],
        "day": [f"{d:02d}" for d in range(1, 32)],
        "daily_statistic": "daily_mean",
        "time_zone": "utc+00:00",
        "frequency": "6_hourly"
    }

    safe_retrieve(
        "derived-era5-single-levels-daily-statistics",
        request,
        target=fpath
    )

    print("Saved:", fpath)
# Loop through all months
# 1972-08 → 2024-06
for y in range(1972, 2025):

    # Start Aug 1972
    if y == 1972:
        start_m = 7
    else:
        start_m = 1

    # End Jun 2024
    if y == 2024:
        end_m = 8
    else:
        end_m = 12

    for m in range(start_m, end_m + 1):
        download_month(y, m)

print("\nALL MONTHS DOWNLOADED SUCCESSFULLY.")


In [ ]:
import numpy as np
import pandas as pd
import os
import xarray as xr
from datetime import datetime, timedelta
import pyreadr
from pyproj import Transformer



ERA5_DIR = r"path/to/snow/data-and-results\snow_era5_monthly"


OUTPUT_CSV = r"path/to/snow/data-and-results\sce_temperature_covariate_fast.csv"


MASK_FILE = r"path/to/snow/data-and-results\mask_cache.npy"

res = pyreadr.read_r("data/snow_cleaned_full.Rda")
snow = res[None]

LON = snow["LON"].values
LAT = snow["LAT"].values

weekly_cols = [c for c in snow.columns if c not in ["LON", "LAT"]]
weekly_dates = [datetime.strptime(c, "%Y-%m-%d") for c in weekly_cols]

num_cells = len(LON)


sample_file = sorted(os.listdir(ERA5_DIR))[0]
sample_ds = xr.open_dataset(os.path.join(ERA5_DIR, sample_file))

lat_grid = sample_ds["latitude"].values
lon_grid = sample_ds["longitude"].values

LATG, LONG = np.meshgrid(lat_grid, lon_grid, indexing="ij")

lat_flat = LATG.ravel()
lon_flat = LONG.ravel()

N_GRID = len(lat_flat)



if not os.path.exists(MASK_FILE):

    print("Generating polygon masks for 1618 cells...")

    to_stereo = Transformer.from_crs("EPSG:4326", "EPSG:3413", always_xy=True)
    GX, GY = to_stereo.transform(lon_flat, lat_flat)

    CX, CY = to_stereo.transform(LON, LAT)

    BOX = 190400  # 190.4km

    masks = np.zeros((num_cells, N_GRID), dtype=bool)

    for i in range(num_cells):
        if i % 100 == 0:
            print("masking cell:", i, "/", num_cells)

        mask = (np.abs(GX - CX[i]) <= BOX/2) & (np.abs(GY - CY[i]) <= BOX/2)
        masks[i, :] = mask

    np.save(MASK_FILE, masks)
    print("Mask saved:", MASK_FILE)

else:
    print("Loading precomputed mask...")
    masks = np.load(MASK_FILE, mmap_mode="r") 




monthly_files = {}
for fname in os.listdir(ERA5_DIR):
    if fname.endswith(".nc"):
        y, m = fname.replace(".nc", "").split("-")
        monthly_files[(int(y), int(m))] = os.path.join(ERA5_DIR, fname)

ds_cache = {}

def load_month_ds(y, m):
    key = (y, m)
    if key not in ds_cache:
        ds_cache[key] = xr.open_dataset(monthly_files[key])
    return ds_cache[key]


def spatial_mean_day(ds, date):
    dd = int(date.strftime("%d")) - 1  # valid_time index
    t2 = ds["t2m"].isel(valid_time=dd).values  # (lat, lon)
    temp_flat = t2.ravel()

    out = np.zeros(num_cells)
    for i in range(num_cells):
        vals = temp_flat[masks[i]]
        out[i] = np.nanmean(vals) if vals.size else np.nan
    return out




def append_week(date_str, arr):
    if not os.path.exists(OUTPUT_CSV):
        df = pd.DataFrame({"LON": LON, "LAT": LAT})
        df[date_str] = arr
        df.to_csv(OUTPUT_CSV, index=False)
    else:
        df = pd.read_csv(OUTPUT_CSV)
        if date_str not in df.columns:
            df[date_str] = arr
            df.to_csv(OUTPUT_CSV, index=False)




for wdate in weekly_dates:

    date_str = wdate.strftime("%Y-%m-%d")

    if os.path.exists(OUTPUT_CSV):
        if date_str in pd.read_csv(OUTPUT_CSV, nrows=1).columns:
            print("Skipping:", date_str)
            continue

    print("\n==== Processing week:", date_str, "====")

    days = [wdate + timedelta(days=k) for k in range(-3, 4)]

    arrs = []
    for d in days:
        ds = load_month_ds(d.year, d.month)
        arr = spatial_mean_day(ds, d)
        arrs.append(arr)

    week_mean = np.nanmean(np.vstack(arrs), axis=0)

    append_week(date_str, week_mean)

print("\nDONE! Output CSV saved at:", OUTPUT_CSV)






In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import os


ERA5_DIR  = r"path/to/snow/data-and-results\snow_era5_monthly"
MASK_FILE = r"path/to/snow/data-and-results\mask_cache.npy"
SCE_CSV   = r"path/to/snow/data-and-results\sce_temperature_covariate_fast.csv"


df = pd.read_csv(SCE_CSV)
LON = df["LON"].values
LAT = df["LAT"].values
masks = np.load(MASK_FILE, mmap_mode="r")

sample_file = sorted(os.listdir(ERA5_DIR))[0]
ds = xr.open_dataset(os.path.join(ERA5_DIR, sample_file))

lat_grid = ds["latitude"].values
lon_grid = ds["longitude"].values


LONG, LATG = np.meshgrid(lon_grid, lat_grid)
lat_flat = LATG.ravel()
lon_flat = LONG.ravel()


cell_i = 100
mask = masks[cell_i]


lon_mask = lon_flat[mask]
lat_mask = lat_flat[mask]


d = 1.5

lon_min = LON[cell_i] - d
lon_max = LON[cell_i] + d
lat_min = LAT[cell_i] - d
lat_max = LAT[cell_i] + d


zoom_mask = (
    (lon_flat >= lon_min) & (lon_flat <= lon_max) &
    (lat_flat >= lat_min) & (lat_flat <= lat_max)
)

lon_zoom = lon_flat[zoom_mask]
lat_zoom = lat_flat[zoom_mask]


plt.figure(figsize=(8, 8))

plt.scatter(lon_zoom, lat_zoom, s=8, color="lightgray", label="ERA5 grid nearby")
plt.scatter(lon_mask, lat_mask, s=25, color="red", label="Mask points")
plt.scatter(LON[cell_i], LAT[cell_i], s=200, color="blue", marker="x", label="Cell center")

plt.xlim(lon_min, lon_max)
plt.ylim(lat_min, lat_max)

plt.title(f"Zoomed Mask Visualization (cell {cell_i})")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.grid()
plt.show()


In [ ]:
import pyreadr
pyreadr.write_rdata("sce_temperature_covariate_fast.Rda", df, df_name="sce_temp")

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import os
from datetime import datetime, timedelta

ERA5_DIR  = r"path/to/snow/data-and-results\snow_era5_monthly"
MASK_FILE = r"path/to/snow/data-and-results\mask_cache.npy"
SCE_CSV   = r"path/to/snow/data-and-results\sce_temperature_covariate_fast.csv"

# Load final output
df = pd.read_csv(SCE_CSV)
LON = df["LON"].values
LAT = df["LAT"].values

# Load masks
masks = np.load(MASK_FILE, mmap_mode="r")

# Preload ERA5 lat/lon grid (mesh)
sample_file = sorted(os.listdir(ERA5_DIR))[0]
ds_sample = xr.open_dataset(os.path.join(ERA5_DIR, sample_file))

lat_grid = ds_sample["latitude"].values
lon_grid = ds_sample["longitude"].values
LONG, LATG = np.meshgrid(lon_grid, lat_grid)
lat_flat = LATG.ravel()
lon_flat = LONG.ravel()


def load_daily_t2m(date):
    """Load specific day's ERA5 t2m (already daily averaged)."""
    fname = f"{date.year}-{date.month:02d}.nc"
    path = os.path.join(ERA5_DIR, fname)
    ds = xr.open_dataset(path)

    day_idx = int(date.strftime("%d")) - 1
    t2 = ds["t2m"].isel(valid_time=day_idx).values  # shape (lat, lon)
    return t2.ravel()  # flatten to match mask


def verify_cell_week(cell_i, week_str):
    week_date = datetime.strptime(week_str, "%Y-%m-%d")
    days = [week_date + timedelta(days=k) for k in range(-3,4)]

    mask_i = masks[cell_i]
    
    daily_means = []
    for d in days:
        arr = load_daily_t2m(d)
        daily_means.append( np.nanmean(arr[mask_i]) )

    manual_avg = np.mean(daily_means)
    final_val = df.loc[cell_i, week_str]

    print("\n========================")
    print("Cell:", cell_i, "Week:", week_str)
    print("7 daily spatial means:", daily_means)
    print("Manual 7-day avg    :", manual_avg)
    print("Final CSV value     :", final_val)
    print("Difference          :", abs(manual_avg - final_val))
    print("========================\n")

    return manual_avg, final_val


verify_cell_week(120, "1980-01-14")

